# Screening COCONUT natural products with the COX-2 model

Run the cells top to bottom. Only **cell 1** needs editing.

**Before you start:** train and save the model first (`cox_training_v3.ipynb`),
then download the COCONUT CSV (lite format is enough) from
https://coconut.naturalproducts.net/download and put it at `COCONUT_CSV`
(see `constants.py`).

**What happens:** the molecules are loaded, scored in chunks by the saved model,
checked for whether they are similar enough to the training data to be trusted,
ranked, and the top hits are explained with TreeSHAP.

**Needs:** `pandas tqdm rdkit shap`

## 1. Settings

In [8]:
# Column names in the COCONUT CSV (printed in the error message if one is wrong)
SMILES_COL = "canonical_smiles"
ID_COL = "identifier"
NAME_COL = "name"

# Start with a random sample to check everything works, None = all molecules
N_MOLECULES = 20000

# Molecules scored per batch (lower = less memory)
CHUNK_SIZE = 5000

# How many top-ranked molecules to list
TOP_N = 50

# How many of those to explain with SHAP
N_EXPLAIN = 3

## 2. Load COCONUT and the model

In [9]:
import ethnohack.coconut as coco
import ethnohack.cox_model.cox_model_api as cox

/home/jenna/workspace/ethnohack/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
cox.model_info()

coconut = coco.load_coconut(
    smiles_col=SMILES_COL, id_col=ID_COL, name_col=NAME_COL, n_max=N_MOLECULES
)
coconut.head()

feature mode : ecfp (2048 columns)
split        : scaffold
cutoff       : 0.40
prevalence  : 0.552
pr_auc      : 0.809
roc_auc     : 0.774
mcc         : 0.391
precision   : 0.721
recall      : 0.745
f1          : 0.733
Loaded 20000 natural products


,id,name,smiles
0,CNP0328306.3,"(~{Z},2~{R})-2-hydroxy-6-[(1~{R},6~{E},9~{S},1...",C/C1=C\[C@H]2O[C@@H](C/C=C/C=CC(=O)O[C@@H]3C[C...
1,CNP0272385.1,NaN,CCCCCCCCCCCCCCCC(=O)O[C@H]1CC[C@]2(C)[C@H]3CC=...
2,CNP0388067.6,NaN,CC(C)C[C@H](NC(=O)[C@@H]1CS[C@@H]2C3=CC=CC=C3C...
3,CNP0213780.21,NaN,CC(C)(O)[C@@H]1CC[C@@](C)([C@H]2[C@@H](O)C[C@@...
4,CNP0268363.2,NaN,CC[C@@H](/C=C\[C@H](C)[C@H]1CC[C@@H]2[C@@H]3CC...


## 3. Score

Includes the reliability check (`max_sim_train`, `in_domain`). This is the slow
part, because every molecule is compared with the whole training set.

In [11]:
scored = coco.score_coconut(coconut, chunk_size=CHUNK_SIZE)
coco.summarize(scored)

Scoring:   0%|          | 0/4 [00:04<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/home/jenna/workspace/ethnohack/src/ethnohack/cox_model/train_smiles_cox.csv'

## 4. Top hits

Only `in_domain` molecules are ranked, because predictions on molecules far
from the training data are much less reliable. The probability is a ranking,
not a verdict: use it to compare and prioritise, not to call a compound active.

In [ ]:
hits = coco.top_hits(scored, n=TOP_N, in_domain_only=True)
hits

In [ ]:
coco.save_scores(scored)

## 5. Explain the top hits (TreeSHAP)

Red = pushes toward active, blue = pushes toward inactive. The waterfall starts
at the average prediction (base value) and ends at the model probability.

In [ ]:
import ethnohack.explainer as xai

model = cox.load_model()
top = hits.head(N_EXPLAIN)

# fingerprint bits get a substructure hint in their name for the plots
feature_names = xai.readable_feature_names(cox.feature_names(), top["smiles"])

for _, row in top.iterrows():
    print(f"\n##### {row['id']} | {row['name']} | model probability: {row['proba']:.3f}")
    xai.explain_molecule(
        row["smiles"],
        model,
        target_class=1,
        feature_names=feature_names,
        featurizer=cox.featurize,
    )